
Input: `outputs/step3_models.joblib`.
Output: `outputs/all_nine_results.csv`, figure `slide_figures/heat.png`.


In [ ]:
# Working folder: holds HABs_ScrippsPier_v3.xlsx and the outputs/ folder shared by all steps.
# In Colab the folder is on Google Drive, so every step notebook can read what the previous step saved.
import os
WORK_FOLDER = "/content/drive/MyDrive/BIF633_Progress2"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(WORK_FOLDER, exist_ok=True)
    os.chdir(WORK_FOLDER)
except ImportError:
    pass  # not running in Colab: use the current folder
os.makedirs("outputs", exist_ok=True)
print("Working folder:", os.getcwd())

In [ ]:
# Load everything saved by the previous step
import joblib
globals().update(joblib.load("outputs/step3_models.joblib"))
print("Loaded outputs/step3_models.joblib")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

# The models predict log(1 + Chl-a). Convert back to mg/m3 before scoring.
def to_mg(log_values):
    return np.clip(np.expm1(np.asarray(log_values)), 0, None)  # same as the team's convert_log_prediction_to_original

def rmse(observed, predicted):
    return np.sqrt(mean_squared_error(observed, predicted))

def mae(observed, predicted):
    return mean_absolute_error(observed, predicted)

# Observed Chl-a (mg/m3) for the training and validation weeks
observed_train = to_mg(y_train)
observed_validation = to_mg(y_validation)

# The three core models and the feature set used for "set C" by each model
core_models = ["Linear Regression", "Random Forest", "SVR"]
set_C_name = {
    "Linear Regression": "C_Linear_Selected",   # 13 features kept after the VIF check
    "Random Forest": "C_Full_Temporal",
    "SVR": "C_Full_Temporal",
}
short_name = {
    "A_Environmental": "A",
    "B_Environmental_Seasonal": "B",
    "C_Full_Temporal": "C",
    "C_Linear_Selected": "C",
}
print("Ready")

In [ ]:
rows = []
for model_name in core_models:
    for feature_set in ["A_Environmental", "B_Environmental_Seasonal", set_C_name[model_name]]:
        result = training_results_df[(training_results_df["Model"] == model_name) &
                                     (training_results_df["Feature_set"] == feature_set)].iloc[0]
        rows.append({"Model": model_name,
                     "Feature set": short_name[feature_set],
                     "RMSE": result["RMSE_original"],
                     "MAE": result["MAE_original"],
                     "RMSE (log scale)": result["RMSE_log"]})

all_nine = pd.DataFrame(rows)
display(all_nine.pivot(index="Model", columns="Feature set", values="RMSE").round(3))
display(all_nine.round(3))

In [ ]:
all_nine.to_csv("outputs/all_nine_results.csv", index=False)
print("Saved outputs/all_nine_results.csv")

In [ ]:
import os
import matplotlib.dates as mdates
os.makedirs("slide_figures", exist_ok=True)  # figures go here
plt.rcParams.update({"font.size": 18, "axes.spines.top": False, "axes.spines.right": False})
PLUM, TEAL, OLIVE, GREY = "#8A3560", "#00958F", "#A4983A", "#B3B0AA"


# Slide 7 (left): validation RMSE of all nine runs
from matplotlib.colors import LinearSegmentedColormap
table = all_nine.pivot(index="Model", columns="Feature set", values="RMSE")
fig, ax = plt.subplots(figsize=(9.6, 4.85))
teal_scale = LinearSegmentedColormap.from_list("teal", ["#0B6F6B", "#7CC2BE", "#EEF4F2"])
ax.imshow(table.clip(4.05, 4.40), cmap=teal_scale, vmin=4.05, vmax=4.40, aspect="auto")
for i in range(table.shape[0]):
    for j in range(table.shape[1]):
        value = table.iloc[i, j]
        best = value == table.iloc[i].min()          # best feature set of each model in bold
        ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=24,
                color="white" if value < 4.13 else "#2B2B2B", fontweight="bold" if best else "normal")
ax.set_xticks(range(3))
ax.set_xticklabels(["A\nEnvironmental", "B\n+ Season", "C\n+ Chl-a history"])
ax.set_yticks(range(3))
ax.set_yticklabels(["Linear\nRegression", "Random\nForest", "SVR"])
ax.tick_params(length=0)
ax.xaxis.tick_top()
for side in ax.spines.values():
    side.set_visible(False)
plt.tight_layout()
plt.savefig("slide_figures/heat.png", dpi=200)
plt.show()